# NHANES Oral Health Disparities Analysis

This notebook supports the Tableau dashboard project:

**Socioeconomic and Behavioral Determinants of Oral Health Outcomes Based on NHANES Data (2013–2018)**

**Project goal:** Analyze how income, education, smoking status, and insurance coverage are associated with oral health outcomes among U.S. adults.

**Dashboard:** NHANES Oral Health Disparities Dashboard

**Main outcomes:**
- Untreated dental caries
- Missing teeth count

**Key predictors:**
- Income-to-poverty ratio
- Education level
- Smoking status
- Dental insurance
- Age
- Sex
- Race/ethnicity


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (9, 5)
sns.set_theme(style="whitegrid")


## 2. Upload and Load Dataset

Upload your cleaned NHANES oral health dataset CSV file.

The dataset should ideally include variables related to:
- Untreated dental caries
- Missing teeth
- Income-to-poverty ratio
- Education
- Smoking status
- Dental insurance
- Age
- Sex
- Race/ethnicity

If your column names are different, update the variable mapping section below.


In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


In [ ]:
csv_files = [file for file in uploaded.keys() if file.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload at least one CSV file.")

file_path = csv_files[0]
df = pd.read_csv(file_path)

print("Loaded file:", file_path)
print("Shape:", df.shape)
df.head()


## 3. Data Inspection

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
df.isnull().sum().sort_values(ascending=False).head(20)


## 4. Standardize Column Names

This step converts column names into analysis-friendly lowercase format.


In [ ]:
data = df.copy()

data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("/", "_")
    .str.replace("__", "_")
)

print(data.columns.tolist())


## 5. Variable Mapping

Update these names if your dataset uses different column labels.

The notebook tries to detect common names automatically.


In [ ]:
def find_col(possible_names, columns):
    col_map = {col.lower(): col for col in columns}
    for name in possible_names:
        if name.lower() in col_map:
            return col_map[name.lower()]
    return None

caries_col = find_col([
    "untreated_dental_caries", "untreated_caries", "caries", "dental_caries", "ohx_caries"
], data.columns)

missing_teeth_col = find_col([
    "missing_teeth", "missing_teeth_count", "tooth_loss", "missing_count"
], data.columns)

income_col = find_col([
    "income_to_poverty_ratio", "income_poverty_ratio", "pir", "poverty_income_ratio", "income"
], data.columns)

education_col = find_col([
    "education", "education_level", "educ"
], data.columns)

smoking_col = find_col([
    "smoking_status", "smoker", "current_smoker", "smoking"
], data.columns)

insurance_col = find_col([
    "dental_insurance", "insurance", "dental_coverage"
], data.columns)

age_col = find_col([
    "age", "ridageyr"
], data.columns)

sex_col = find_col([
    "sex", "gender", "riagendr"
], data.columns)

race_col = find_col([
    "race_ethnicity", "race", "ethnicity", "ridreth1", "ridreth3"
], data.columns)

mapped_columns = {
    "Untreated caries": caries_col,
    "Missing teeth": missing_teeth_col,
    "Income-to-poverty ratio": income_col,
    "Education": education_col,
    "Smoking status": smoking_col,
    "Dental insurance": insurance_col,
    "Age": age_col,
    "Sex": sex_col,
    "Race/ethnicity": race_col
}

mapped_columns


## 6. Data Cleaning

This section:
- Converts numeric columns to numeric type
- Standardizes categorical labels
- Creates income categories for visualization


In [ ]:
# Clean object columns
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype(str).str.strip()
    data[col] = data[col].replace(["nan", "NaN", "None", "", "?"], np.nan)

# Convert likely numeric columns
for col in [caries_col, missing_teeth_col, income_col, age_col]:
    if col and col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# Create income categories
if income_col:
    data["income_category"] = pd.cut(
        data[income_col],
        bins=[-np.inf, 1.3, 3.5, np.inf],
        labels=["Low income", "Middle income", "High income"]
    )

print("Cleaned dataset shape:", data.shape)
data.head()


## 7. Descriptive Statistics

In [ ]:
summary_numeric = data.select_dtypes(include=["int64", "float64"]).describe().T
summary_numeric


## 8. Oral Health Outcome by Income Level

In [ ]:
if caries_col and "income_category" in data.columns:
    caries_by_income = (
        data.groupby("income_category")[caries_col]
        .mean()
        .reset_index()
    )
    caries_by_income["caries_prevalence_percent"] = caries_by_income[caries_col] * 100

    sns.barplot(data=caries_by_income, x="income_category", y="caries_prevalence_percent")
    plt.title("Prevalence of Untreated Dental Caries by Income Level")
    plt.xlabel("Income Category")
    plt.ylabel("Caries Prevalence (%)")
    plt.show()

    caries_by_income
else:
    print("Caries column or income category not found.")


## 9. Missing Teeth by Education Level

In [ ]:
if missing_teeth_col and education_col:
    missing_by_education = (
        data.groupby(education_col)[missing_teeth_col]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    sns.barplot(data=missing_by_education, x=missing_teeth_col, y=education_col)
    plt.title("Average Missing Teeth by Education Level")
    plt.xlabel("Average Missing Teeth")
    plt.ylabel("Education Level")
    plt.show()

    missing_by_education
else:
    print("Missing teeth column or education column not found.")


## 10. Oral Health Outcome by Smoking Status

In [ ]:
if caries_col and smoking_col:
    smoking_caries = (
        data.groupby(smoking_col)[caries_col]
        .mean()
        .reset_index()
    )
    smoking_caries["caries_prevalence_percent"] = smoking_caries[caries_col] * 100

    sns.barplot(data=smoking_caries, x=smoking_col, y="caries_prevalence_percent")
    plt.title("Untreated Dental Caries Prevalence by Smoking Status")
    plt.xlabel("Smoking Status")
    plt.ylabel("Caries Prevalence (%)")
    plt.xticks(rotation=30)
    plt.show()

    smoking_caries
else:
    print("Caries column or smoking column not found.")


## 11. Missing Teeth by Dental Insurance Status

In [ ]:
if missing_teeth_col and insurance_col:
    insurance_missing = (
        data.groupby(insurance_col)[missing_teeth_col]
        .mean()
        .reset_index()
    )

    sns.barplot(data=insurance_missing, x=insurance_col, y=missing_teeth_col)
    plt.title("Average Missing Teeth by Dental Insurance Status")
    plt.xlabel("Dental Insurance")
    plt.ylabel("Average Missing Teeth")
    plt.xticks(rotation=30)
    plt.show()

    insurance_missing
else:
    print("Missing teeth column or insurance column not found.")


## 12. Logistic Regression: Predicting Untreated Dental Caries

This model predicts untreated dental caries using socioeconomic and behavioral predictors.


In [ ]:
predictor_cols = [
    income_col,
    education_col,
    smoking_col,
    insurance_col,
    age_col,
    sex_col,
    race_col
]

predictor_cols = [col for col in predictor_cols if col and col in data.columns]

if not caries_col:
    raise ValueError("Untreated caries column not found. Update caries_col mapping.")

model_df = data[predictor_cols + [caries_col]].dropna(subset=[caries_col]).copy()

# Keep binary values only if possible
model_df = model_df[model_df[caries_col].isin([0, 1])]

X = model_df[predictor_cols]
y = model_df[caries_col].astype(int)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y if y.nunique() == 2 else None
)

log_model.fit(X_train, y_train)
preds = log_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("\nClassification Report:")
print(classification_report(y_test, preds))

cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix: Untreated Dental Caries")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 13. Logistic Regression Odds Ratios

In [ ]:
# Extract odds ratios from logistic regression

feature_names = []

if numeric_features:
    feature_names.extend(numeric_features)

if categorical_features:
    ohe = log_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
    cat_names = ohe.get_feature_names_out(categorical_features).tolist()
    feature_names.extend(cat_names)

coefs = log_model.named_steps["classifier"].coef_[0]
odds_ratios = np.exp(coefs)

odds_ratio_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefs,
    "Odds_Ratio": odds_ratios
}).sort_values(by="Odds_Ratio", ascending=False)

odds_ratio_df.head(20)


In [ ]:
top_or = odds_ratio_df.head(15)

sns.barplot(data=top_or, x="Odds_Ratio", y="Feature")
plt.axvline(1, linestyle="--")
plt.title("Top Odds Ratios for Untreated Dental Caries")
plt.xlabel("Odds Ratio")
plt.ylabel("Feature")
plt.show()


## 14. Linear Regression: Predicting Missing Teeth Count

This model estimates missing teeth counts using socioeconomic and behavioral predictors.


In [ ]:
if not missing_teeth_col:
    print("Missing teeth column not found. Skipping linear regression.")
else:
    linear_df = data[predictor_cols + [missing_teeth_col]].dropna(subset=[missing_teeth_col]).copy()

    X_lin = linear_df[predictor_cols]
    y_lin = linear_df[missing_teeth_col]

    numeric_features_lin = X_lin.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features_lin = X_lin.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_transformer_lin = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer_lin = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor_lin = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_lin, numeric_features_lin),
            ("cat", categorical_transformer_lin, categorical_features_lin)
        ]
    )

    lin_model = Pipeline(steps=[
        ("preprocessor", preprocessor_lin),
        ("regressor", LinearRegression())
    ])

    X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
        X_lin, y_lin, test_size=0.30, random_state=42
    )

    lin_model.fit(X_train_lin, y_train_lin)
    lin_preds = lin_model.predict(X_test_lin)

    print("MAE:", round(mean_absolute_error(y_test_lin, lin_preds), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test_lin, lin_preds)), 4))
    print("R2:", round(r2_score(y_test_lin, lin_preds), 4))


## 15. Dashboard Output Tables

In [ ]:
output_dir = Path("nhanes_oral_health_outputs")
output_dir.mkdir(exist_ok=True)

data.to_csv(output_dir / "cleaned_nhanes_oral_health_data.csv", index=False)

if caries_col and "income_category" in data.columns:
    caries_by_income.to_csv(output_dir / "caries_by_income.csv", index=False)

if missing_teeth_col and education_col:
    missing_by_education.to_csv(output_dir / "missing_teeth_by_education.csv", index=False)

if caries_col and smoking_col:
    smoking_caries.to_csv(output_dir / "caries_by_smoking_status.csv", index=False)

if missing_teeth_col and insurance_col:
    insurance_missing.to_csv(output_dir / "missing_teeth_by_insurance.csv", index=False)

odds_ratio_df.to_csv(output_dir / "logistic_regression_odds_ratios.csv", index=False)

print("Saved output files:")
for f in output_dir.iterdir():
    print("-", f)


## 16. Key Findings

Based on the capstone poster and dashboard interpretation:

- Untreated dental caries prevalence was higher among low-income adults than high-income adults.
- Missing teeth counts were higher among less educated adults and current smokers.
- Lack of dental insurance was associated with more missing teeth.
- Smoking and poverty were strong predictors of poor oral health outcomes.
- Income, education, and smoking collectively explained meaningful variation in oral health outcomes.


## 17. Public Health Implications

Expanding preventive dental coverage, improving oral-health literacy, and integrating smoking cessation into community programs may help reduce oral health disparities and promote health equity.


## 18. Conclusion

This notebook demonstrates a health equity analytics workflow using NHANES oral health data. It combines data cleaning, descriptive analysis, bivariate visualization, logistic regression, linear regression, and dashboard-ready outputs to support evidence-based oral-health policy and intervention planning.
